# 2장 01. score와 threshold 확인

이 notebook은 모델이 만든 `score`가 `threshold`를 지나 `prediction`이 되는 흐름만 봅니다.


## 기본 개념: score, threshold, prediction

Score는 모델이 각 sample에 대해 내는 연속적인 출력입니다. 이 과정에서는 `high_risk` class 쪽 가능성을 나타내는 값으로 사용하지만, score를 항상 실제 확률이라고 해석하면 안 됩니다. Calibration이 되어 있지 않으면 0.8이라는 score가 실제 80% 확률이라는 뜻은 아닐 수 있습니다.

Threshold는 score를 운영 class로 바꾸는 기준입니다. 같은 모델과 같은 score라도 threshold가 0.3인지 0.5인지에 따라 prediction 분포와 FP/FN 균형이 달라집니다. 그래서 MLOps에서는 모델 artifact와 threshold를 함께 버전 관리해야 합니다.

Prediction은 API나 batch job이 외부 시스템에 전달하는 최종 결과입니다. 운영 장애 조사에서는 score 분포가 변했는지, threshold가 바뀌었는지, prediction 집계만 바뀌었는지를 분리해야 합니다.

| 용어 | 의미 | 운영에서 확인할 위치 |
| --- | --- | --- |
| `score` | threshold 적용 전 모델 출력 | model output, log, monitoring |
| `threshold` | score를 class로 바꾸는 기준 | config, MLflow param, deployment env |
| `prediction` | 외부에 전달되는 최종 class | API response, batch output, log |
| score distribution | score가 어느 구간에 몰리는지 | model monitoring dashboard |

이 노트북은 성능 향상을 다루지 않습니다. MLOps에서 자주 생기는 “모델이 바뀐 것인가, threshold가 바뀐 것인가, 입력이 바뀐 것인가”라는 질문을 분리하기 위한 최소 개념을 다룹니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
dataframe, source = aiq.load_csv_or_sample(
    "data/vital_signs_test.csv",
    aiq.sample_vital_signs(),
    nrows=30,
)
print("source:", source)
print("rows:", len(dataframe))


## 1. score 만들기

여기서는 Lite용 단순 scoring rule을 사용합니다. 목표는 모델 성능 향상이 아니라 score 흐름을 보는 것입니다.


In [ ]:
work = dataframe.copy()
work["label"] = work["label"].map(aiq.normalize_label)
work["score"] = aiq.score_rows(work)
work[["patient_id", "label", "score"]].head(10)


## 2. threshold 적용하기

`score`가 threshold 이상이면 `high_risk`, 아니면 `low_risk`로 바꿉니다.


In [ ]:
threshold = aiq.THRESHOLD
work["threshold"] = threshold
work["prediction"] = "low_risk"
work.loc[work["score"] >= threshold, "prediction"] = "high_risk"

work[["patient_id", "score", "threshold", "prediction", "label"]].head(10)
